In [ ]:
import Pkg
Pkg.update()
Pkg.add("ProfileView")
Pkg.add("ProfileCanvas")

In [ ]:
import Pkg
using MLDatasets
using Flux: onehotbatch, onecold
using Statistics
using LinearAlgebra
using Random
using ProfileCanvas
using InteractiveUtils

train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Sizes: ", size(X_train))

include("clothesolver.jl") 

my_net_def = Chain(
  Conv((3, 3), 1 => 6, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),               
  Dense(784 => 84, relu),
  Dropout(0.4),
  Dense(84 => 10)
)

my_model = build_model(my_net_def, (28, 28, 1))
input_node = alloc_act!(my_model.pool, 28, 28, 1)
target_node = alloc_act!(my_model.pool, 10)
loss_fn = LogitCrossEntropy(my_model.pool, 10)

In [ ]:
bs = my_model.batch_size

x_flat = parent(input_node.data)
y_flat = parent(target_node.data)

len_x_batch = 28 * 28 * 1 * bs
len_y_batch = 10 * bs

@inbounds for k in 1:len_x_batch
    x_flat[k] = X_train[k]
end
@inbounds for k in 1:len_y_batch
    y_flat[k] = Y_train[k]
end

preds = forward_train!(my_model.chain, input_node)
primal!(loss_fn, preds, target_node)
loss_fn.out.grad[1] = 1.0f0
adjoint!(loss_fn, preds, target_node)
backward!(my_model.chain, input_node)

println("Compilation finished, starting profiling...")

@time ProfileCanvas.@profview for _ in 1:1000
    zero_a_grad!(my_model.pool)
    zero_w_grad!(my_model.pool)
    
    @inbounds for k in 1:len_x_batch
        x_flat[k] = X_train[k]
    end
    @inbounds for k in 1:len_y_batch
        y_flat[k] = Y_train[k]
    end
    
    preds = forward_test!(my_model.chain, input_node)
    primal!(loss_fn, preds, target_node)
    
    loss_fn.out.grad[1] = 1.0f0
    adjoint!(loss_fn, preds, target_node)
    backward!(my_model.chain, input_node)
end

In [ ]:
println("============== All Layers ==============")

curr_x = my_model.input 

for (i, layer) in enumerate(my_model.chain.layers)
    layer_name = typeof(layer).name.name
    println("\n\n" * "="^10 * " Layer $i: $layer_name " * "="^10)
    code_warntype(primal!, (typeof(layer), typeof(curr_x)))
    curr_x = layer.out 
end